In [1]:
# Libraries
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import re

# Add the parent directory to path
sys.path.append("..")

# Custom utils
from utils.json import load_json, save_json

In [2]:
def json_structure(data, max_depth=5, current_depth=0, path="root"):
    """
    Recursively explore and print the structure of JSON data up to a specified depth
    
    Args:
        data: The JSON data to explore
        max_depth: Maximum depth to explore
        current_depth: Current depth in the recursion
        path: Current path in the JSON structure
    """
    
    # Indentation for visualizing depth
    indent = (2 * " ") * current_depth
    
    if current_depth >= max_depth:
        print(f"{indent}[Reached max depth at {path}]")
        return
    
    # Handle dictinary
    if isinstance(data, dict):
        print(f"{indent}{path} (dict with {len(data)} keys)")
        for key, value in data.items():
            new_path = f"{path}.{key}" if path != "root" else key
            json_structure(value, max_depth, current_depth + 1, new_path)
    
    # Handle list
    elif isinstance(data, list):
        print(f"{indent}{path} (list with {len(data)} items)")
        if data and current_depth < max_depth - 1:
            # Show structure of first item as an example
            sample_item = data[0]
            new_path = f"{path}[0]"
            json_structure(sample_item, max_depth, current_depth + 1, new_path)
            
            # If there are multiple different structures in the list, show another example
            if len(data) > 1 and not all(type(item) == type(sample_item) for item in data):
                different_type_item = next((item for item in data if type(item) != type(sample_item)), None)
                if different_type_item:
                    new_path = f"{path}[different_type]"
                    json_structure(different_type_item, max_depth, current_depth + 1, new_path)
    
    # Handle other types
    elif isinstance(data, (str, int, float, bool, type(None))):
        # For primitive types, show a sample value
        if isinstance(data, str) and len(data) > 30:
            sample_value = f"{data[:30]}..." 
        else:
            sample_value = data
        print(f"{indent}{path} ({type(data).__name__}): {sample_value}")

In [3]:
def json_keys(data):
    """
    Extract the key structural components of the dataset
    
    Returns:
        dict: Dictionary containing the structural information
    """
    structure = {
        "main_keys": set(),
        "data_keys": set(),
        "prediction_keys": set(),
        "result_keys": set(),
        "value_keys": set(),
        "label_types": set()
    }
    
    # Process documents to extract generic structure
    for doc in data[:15]:
        # Main level keys
        for key in doc.keys():
            structure["main_keys"].add(key)
        
        # Data level keys
        if "data" in doc:
            for key in doc["data"].keys():
                structure["data_keys"].add(key)
        
        # Prediction level
        if "predictions" in doc and doc["predictions"]:
            for pred in doc["predictions"]:
                for key in pred.keys():
                    structure["prediction_keys"].add(key)
                
                # Result level
                if "result" in pred:
                    for res in pred["result"]:
                        for key in res.keys():
                            structure["result_keys"].add(key)
                        
                        # Value level and labels
                        if "value" in res:
                            for key in res["value"].keys():
                                structure["value_keys"].add(key)
                            
                            # Collect label types
                            if "labels" in res["value"]:
                                for label in res["value"]["labels"]:
                                    structure["label_types"].add(label)
    
    # Convert sets to sorted lists for nicer output
    for key in structure:
        structure[key] = sorted(list(structure[key]))
    
    return structure
    

In [4]:
# Load the training data
path = "../data/raw/negacio_train_v2024.json"
data = load_json(path)

In [5]:
print(f"Type of loaded data: {type(data)}")
print(f"Number of documents: {len(data)}")

Type of loaded data: <class 'list'>
Number of documents: 254


In [6]:
# Explore your JSON data
json_structure(data, max_depth=10)

root (list with 254 items)
  root[0] (dict with 3 keys)
    root[0].data (dict with 6 keys)
      root[0].data.cmbd (str): null
      root[0].data.id (str): 19026587
      root[0].data.docid (str): null
      root[0].data.page (str): null
      root[0].data.paragraph (str): null
      root[0].data.text (str):  nº historia clinica: ** *** *...
    root[0].annotations (list with 0 items)
    root[0].predictions (list with 1 items)
      root[0].predictions[0] (dict with 1 keys)
        root[0].predictions[0].result (list with 20 items)
          root[0].predictions[0].result[0] (dict with 5 keys)
            root[0].predictions[0].result[0].value (dict with 3 keys)
              root[0].predictions[0].result[0].value.start (int): 449
              root[0].predictions[0].result[0].value.end (int): 452
              root[0].predictions[0].result[0].value.labels (list with 1 items)
                root[0].predictions[0].result[0].value.labels[0] (str): NEG
            root[0].predictions[0]

In [ ]:
# Get the JSON keys structure
structure = json_keys(data)

display(structure)

{'main_keys': ['annotations', 'data', 'predictions'],
 'data_keys': ['cmbd', 'docid', 'id', 'page', 'paragraph', 'text'],
 'prediction_keys': ['result'],
 'result_keys': ['from_name', 'id', 'to_name', 'type', 'value'],
 'value_keys': ['end', 'labels', 'start'],
 'label_types': ['NEG', 'NSCO', 'UNC', 'USCO']}

In [23]:
# Insights using the first document
record_first = data[0]

if isinstance(data, list):
    print(f"Type document: {type(data[0])}")
    print(f"Keys document: {structure['main_keys']}")
    print()

    for main_keys in record_first:
        print(f"Key: {main_keys}")
        print(f"Type: {type(record_first[main_keys])}")
        print(f"Length: {len(record_first[main_keys])}")
        print(f"Value: {record_first[main_keys]}")
        print()

        if "data" in main_keys:
            for k in record_first["data"].keys():
                print(f"\tKey: {k}")
                print(f"\tType: {type(record_first['data'][k])}")
                print(f"\tLength: {len(record_first['data'][k])}")
                print(f"\tValue: {record_first['data'][k]}")
                print()

Type document: <class 'dict'>
Keys document: ['annotations', 'data', 'predictions']

Key: data
Type: <class 'dict'>
Length: 6
Value: {'cmbd': 'null', 'id': '19026587', 'docid': 'null', 'page': 'null', 'paragraph': 'null', 'text': " nº historia clinica: ** *** *** nºepisodi: ******** sexe: home data de naixement: 16.05.1936 edat: 82 anys procedencia cex mateix hosp servei urologia data d'ingres 24.07.2018 data d'alta 25.07.2018 08:54:04 ates per ***************, *****; ****************, ****** informe d'alta d'hospitalitzacio motiu d'ingres paciente que ingresa de forma programada para realizacion de uretrotomia interna . antecedents alergia a penicilina y cloramfenicol . no habitos toxicos. antecedentes medicos: bloqueo auriculoventricular de primer grado hipertension arterial. diverticulosis extensa insuficiencia renal cronica colelitiasis antecedentes quirurgicos: exeresis de lesiones cutaneas con anestesia local protesis total de cadera cordectomia herniorrafia inguinal proces actua